# Experiment runner: stretch goals (GATv2 and low-homophily graphs)
Runs the proposal's stretch experiments on a Colab GPU (T4), using the same code as the main runner
(<https://github.com/YashDolorase/cs545-gat>).

| Experiment | Setup | Runs |
|---|---|---|
| GATv2 benchmark | GATv2 on Cora and CiteSeer | 2 datasets × 10 seeds |
| GATv2 under noise | GATv2 on Cora with 0 / 10 / 25 / 50 / 100 % random edges | 5 noise levels × 10 seeds |
| Low homophily | MLP, GCN, GraphSAGE, GAT, GATv2 on Texas and Chameleon | 5 models × 2 datasets × 10 splits |

**GATv2** (Brody et al., 2022) applies the nonlinearity before the attention vector:
$e_{ij} = \mathbf a^	op \mathrm{LeakyReLU}(W[h_i \Vert h_j])$. GAT's attention is *static*: every node ranks its
neighbours in the same order. GATv2's attention is *dynamic*: the ranking can depend on the node doing the attending.

On **low-homophily** graphs, most edges join nodes of different classes. Uniform neighbour averaging mixes in
features from the wrong class, so these graphs are where learned attention should help most.

**Reproducibility.** Seeds fix the initialisation, dropout and noisy graphs, but some GPU scatter operations are not deterministic, so a rerun can differ slightly from the committed `results/`. In our rerun, means moved by at most 0.5 points and no conclusion changed.

## Setup

In [ ]:
import os, torch

# Clone the project repository and install PyTorch Geometric (Colab ships PyTorch but not PyG)
PROJECT = "/content/project"
if not os.path.exists(PROJECT):
    !git clone -q https://github.com/YashDolorase/cs545-gat.git {PROJECT}
os.chdir(PROJECT)
!pip -q install torch_geometric

import torch_geometric
print(f"torch {torch.__version__} | PyG {torch_geometric.__version__} | "
      f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Results directory: starts empty, so every run trains from scratch (the runner skips runs already in its CSV).
# Google Drive is used when available, so the results survive a disconnect.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/CS545_GAT/fresh_stretch"
except Exception:
    OUT = "/content/fresh_stretch"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT, "| existing files:", os.listdir(OUT) or "none")

## The datasets: how homophilous are they?
Edge homophily is the fraction of edges that join two nodes of the same class.
Texas (WebKB) and Chameleon (Wikipedia) come with 10 fixed train/val/test splits. Run *i* uses split *i*.

In [ ]:
import pandas as pd
from src.data import load_dataset, describe, select_split

# Load all four graphs and print their size, number of classes and edge homophily.
# Texas and Chameleon store 10 splits; select_split keeps split 0 so train/val/test counts are for a single split.
stats = [describe(name, select_split(load_dataset(name, "data")[1], 0)) for name in ["Cora", "CiteSeer", "Texas", "Chameleon"]]
display(pd.DataFrame(stats).set_index("dataset"))

## Low-homophily benchmark: Texas and Chameleon
All five models, each with its paper hyperparameters (as in E1), on each of the 10 splits. Results → `e1_benchmark.csv`.

In [ ]:
os.chdir(PROJECT)  # Colab resets the working directory after a reconnect
!python -W ignore scripts/run_experiments.py --exp e1 --models MLP GCN SAGE GAT GATv2 --datasets Texas Chameleon --out {OUT}

## GATv2 on Cora and CiteSeer
Same protocol and seeds as E1, so these runs can be paired with the GAT, GCN and GraphSAGE runs from the main runner.
Results → `e1_benchmark.csv`.

In [ ]:
os.chdir(PROJECT)
!python -W ignore scripts/run_experiments.py --exp e1 --models GATv2 --datasets Cora CiteSeer --out {OUT}

## GATv2 under noise
Same noisy Cora graphs as E3: each seed adds the same random edges as in the main runner.
Results, including GATv2's attention on injected and original edges, → `e3_noise.csv`.

In [ ]:
os.chdir(PROJECT)
!python -W ignore scripts/run_experiments.py --exp e3 --models GATv2 --out {OUT}

## Results summary

In [ ]:
# Stretch results; the main runner's results are added to the same tables when they are in the sibling folder
MAIN = os.path.join(os.path.dirname(OUT), "fresh_run")
def load(name):
    df = pd.read_csv(f"{OUT}/{name}")
    if os.path.exists(f"{MAIN}/{name}"):
        df = pd.concat([df, pd.read_csv(f"{MAIN}/{name}")], ignore_index=True)
    return df

# Test accuracy (%) as mean ± std over seeds/splits
e1 = load("e1_benchmark.csv")
s = e1.groupby(["dataset", "model"])["test_acc"].agg(["mean", "std"])
s = (100 * s["mean"]).round(1).astype(str) + " ± " + (100 * s["std"]).round(1).astype(str)
display(s.unstack("dataset")[[d for d in ["Cora", "CiteSeer", "Texas", "Chameleon"] if d in e1.dataset.unique()]])

# Noisy Cora: mean test accuracy (%) per noise level (rows) and model (columns)
e3 = load("e3_noise.csv")
display((100 * e3.pivot_table(index="noise", columns="model", values="test_acc", aggfunc="mean")).round(1))

# Attention on injected vs original edges relative to uniform (1.0 = what uniform weights would give)
att = ["L1_att_original", "L1_att_injected", "L2_att_original", "L2_att_injected"]
display(e3[e3.model.isin(["GAT", "GATv2"]) & (e3.noise > 0)].groupby(["model", "noise"])[att].mean().round(3))

In [ ]:
# Package the results directory for download
!cd {os.path.dirname(OUT)} && zip -qr /content/fresh_stretch.zip {os.path.basename(OUT)}
from google.colab import files
files.download("/content/fresh_stretch.zip")